In [1]:
import roboticstoolbox as rtb
from spatialmath import SE3
import numpy as np
import os
import swift
import time

## Load Robot from URDF

In [2]:
robot = rtb.ERobot.URDF(os.path.abspath("robot.urdf"))

print(robot)
for i, link in enumerate(robot.links):
    if link.isjoint:
        ql = link.qlim
        print(f"Joint {i} ({link.name}): [{np.degrees(ql[0]):.1f}°, {np.degrees(ql[1]):.1f}°]")

ERobot: robot, 6 joints (RRRRRR), dynamics, geometry
┌──────┬───────────┬───────┬───────────┬───────────────────────────────────────────────┐
│ link │   link    │ joint │  parent   │              ETS: parent to link              │
├──────┼───────────┼───────┼───────────┼───────────────────────────────────────────────┤
│    0 │ base_link │       │ BASE      │ SE3()                                         │
│    1 │ link0     │       │ base_link │ SE3()                                         │
│    2 │ link1     │     0 │ link0     │ SE3(0, 0, 0.065) ⊕ Rz(q0)                     │
│    3 │ link2     │     1 │ link1     │ SE3(0, 0.041, 0.042; -90°, -0°, 0°) ⊕ Rz(q1)  │
│    4 │ link3     │     2 │ link2     │ SE3(0, -0.098, 0.001; 180°, -0°, 0°) ⊕ Rz(q2) │
│    5 │ link4     │     3 │ link3     │ SE3(0, 0.041, 0.042; -90°, -0°, 0°) ⊕ Rz(q3)  │
│    6 │ link5     │     4 │ link4     │ SE3(0, 0.041, 0.057; -90°, -0°, 0°) ⊕ Rz(q4)  │
│    7 │ link6     │     5 │ link5     │ SE3(0, -0.041, 0

## Trajectory in Cartesian Space

In [3]:
start = SE3(0.12, 0.12, 0.09) * SE3.RPY(np.pi, 0, 0)
end   = SE3(0.12, 0.12, 0.01) * SE3.RPY(np.pi, 0, 0)

traj = rtb.ctraj(start, end, 50)

q_prev = robot.ikine_LM(start, tol=0.001, seed=100).q
sols = []
for T in traj:
    sol = robot.ikine_LM(T, q0=q_prev, tol=0.001, seed=100)
    if not sol.success:
        print("IK failed at pose:")
        print(T)
        continue
    q_prev = sol.q
    sols.append(sol.q)
sols = np.array(sols)

env = swift.Swift()
env.launch(browser='notebook')
env.add(robot)

for _ in range(10):
    for q in sols:
        robot.q = q
        env.step(0.05)
        time.sleep(0.05)